# VQE: H2 Ground State (Braket)

Variational Quantum Eigensolver for the H2 ground state using Braket's `LocalSimulator`, a hardware-efficient ansatz, and COBYLA optimisation.

In [ ]:
import numpy as np
from braket.circuits import Circuit, ResultType
from braket.devices import LocalSimulator
import scipy.optimize as opt

## H2 Hamiltonian

In [ ]:
I2 = np.eye(2, dtype=complex)
PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)

def pauli_string(ops):
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

H2_MATRIX = (
    -0.81261 * pauli_string([I2, I2])
    + 0.17120 * pauli_string([PAULI_Z, I2])
    - 0.22279 * pauli_string([I2, PAULI_Z])
    + 0.17120 * pauli_string([PAULI_Z, PAULI_Z])
    + 0.04532 * pauli_string([PAULI_X, PAULI_X])
)

EXACT_GS_ENERGY = -1.380398
N_LAYERS = 3
N_PARAMS = 4 * N_LAYERS

eigenvalues = np.linalg.eigvalsh(H2_MATRIX)
print(f"Exact eigenvalues: {np.round(eigenvalues, 6)}")
print(f"Ground state energy: {eigenvalues[0]:.6f}")

## Ansatz and simulation

In [ ]:
def ansatz_circuit(params):
    circuit = Circuit()
    for layer in range(N_LAYERS):
        base = layer * 4
        circuit.ry(0, params[base + 0])
        circuit.rz(0, params[base + 1])
        circuit.ry(1, params[base + 2])
        circuit.rz(1, params[base + 3])
        circuit.cnot(0, 1)
    return circuit

def simulate(circuit):
    circuit.add_result_type(ResultType.StateVector())
    device = LocalSimulator()
    task = device.run(circuit, shots=0)
    return np.array(task.result().result_types[0].value, dtype=complex)

def energy(params):
    psi = simulate(ansatz_circuit(params))
    return float(np.real(psi.conj() @ H2_MATRIX @ psi))

## COBYLA optimisation

In [ ]:
rng = np.random.default_rng(42)
init_params = rng.uniform(0, 2 * np.pi, size=N_PARAMS)
print(f"Initial energy: {energy(init_params):.6f}")

history = []
def callback(xk):
    history.append(energy(xk))

result = opt.minimize(
    energy, init_params, method="COBYLA",
    options={"maxiter": 150, "rhobeg": 0.5}, callback=callback,
)

print(f"\nOptimised energy: {result.fun:.6f}")
print(f"Error vs exact:   {abs(result.fun - EXACT_GS_ENERGY):.6f}")

## Energy convergence and final state

In [ ]:
step = max(1, len(history) // 10)
for i in range(0, len(history), step):
    print(f"  iter {i + 1:>3d}  energy = {history[i]:.6f}")
if (len(history) - 1) % step != 0:
    print(f"  iter {len(history):>3d}  energy = {history[-1]:.6f}")

psi_final = simulate(ansatz_circuit(result.x))
probs = np.abs(psi_final) ** 2
print("\nFinal state probabilities:")
for i in range(4):
    if probs[i] > 0.001:
        print(f"  |{i:02b}>  P = {probs[i]:.6f}")
print("\nOptimised circuit:")
print(ansatz_circuit(result.x))